<a href="https://colab.research.google.com/github/eTcilopp/temp_python_intro_course_gb/blob/master/Devops_script_finder_content.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
import requests
import json
from tqdm import tqdm
import re

Get list of workspaces from DEVOPS

In [ ]:
url = "http://devops.vitruvi.cc/api/v1/workspaces/"

payload = {}
headers = {
  'Content-Type': 'application/json',
  'Authorization': f'Basic {userdata.get("vitruvi_devops_login")})'
}

response = requests.request("GET", url, headers=headers, data=payload)
workspace_data = response.json().get('results')


Verifying, that workspaces are valid

In [ ]:
viewed_workspaces_lst = []
valid_workspaces_lst = []
errored_workspaces_lst = []
for workspace_record in workspace_data:
    workspace_name = workspace_record.get('name')

    viewed_workspaces_lst.append(workspace_name)
    url = f"https://{workspace_name}.vitruvi.cc/"
    try:
      response = requests.head(url, timeout=5)
    except Exception:
      errored_workspaces_lst.append(workspace_name)
      continue
    if response.status_code != 200:
      errored_workspaces_lst.append(workspace_name)
      continue
    valid_workspaces_lst.append(workspace_name)

Scanning workspaces for script by name

In [ ]:
TARGET_SCRIPT_CONTENT = [
    # --- Datetime & timezone ---
    r"datetime\.utcnow\(",          # deprecated, use datetime.now(UTC)
    r"datetime\.now\(",             # check for naive datetime usage
    r"datetime\.fromisoformat\(",   # behavior changed
    r"timezone\.utc",               # django.utils.timezone.utc removed
    r"django\.utils\.timezone\.utc",
    r"pytz",                        # pytz removed, use zoneinfo
    r"is_dst",                      # arg removed
    r"django\.utils\.datetime_safe",

    # --- Regex migration ---
    r"\(\?i\)\(null\|none\)\\b",    # must become (?i:(null|none))\b

    # --- Numeric / random ---
    r"-1e[0-9]+",                   # scientific notation negatives → float
    r"randint\(",                   # randint args must be ints

    # --- Django ORM / Queryset ---
    r"nulls_first\s*=",             # False/True no longer valid
    r"nulls_last\s*=",
    r"\.iterator\(",                # must specify chunk_size if prefetch used

    # --- Removed functions / utils ---
    r"make_random_password\(",      # use random_password
    r"random_password\(",           # check correct replacement usage
    r"django\.core\.files\.File",   # now requires filename
    r"get_storage_class\(",         # removed
    r"CIEmailField",                # removed
    r"CITextField",

    # --- Distutils removal ---
    r"import\s+distutils",
    r"from\s+setuptools\s+import\s+distutils",
    r"distutils\.",

    # --- Other removals ---
    r"CIEmail",
    r"CIText",
]


In [ ]:
def get_number_of_executions(script_id: int, start_datetime: str, base_url: str, headers: dict, timeout=5):
  GET_SCRIPT_EXECUTION_STATS = f"raw/async_events/scriptexecution/?script_id={script_id}&timestamp__gte={start_datetime}"
  url = f"{base_url}/api/v1/{GET_SCRIPT_EXECUTION_STATS}"

  try:
    response = requests.request("GET", url, headers=headers, timeout=timeout)
    return response.json().get('count')
  except Exception:
    return 0

In [ ]:
def find_matches(code: str, patterns: list[str]) -> list[str]:
    """Return all patterns that match the given code (supports regex or plain substring)."""
    matches = []
    for pattern in patterns:
        try:
            # Try regex search, case-insensitive
            if re.search(pattern, code, re.IGNORECASE):
                matches.append(pattern)
        except re.error:
            # If not a valid regex, fallback to substring match
            if pattern.lower() in code.lower():
                matches.append(pattern)
    return matches

In [ ]:
headers = {
  'Content-Type': 'application/json',
  'Authorization': f'Basic {userdata.get("vitruvi_generic_login")}'
}

stat_start_datetime = '2025-04-01T00:00:00-03:00'
target_workspace_lst = []
failed_workspaces_lst = []
target_script_with_author_lst = []
for workspace_name in tqdm(valid_workspaces_lst, desc="Checking Workspaces"):
    # print(f'Checking Workspace `{workspace_name}`')
    base_url = f"https://{workspace_name}.api.vitruvi.cc"
    url = f"{base_url}/api/v1/raw/async_events/script"
    try:
        response = requests.request("GET", url, headers=headers, timeout=5)
    except Exception as e:
        failed_workspaces_lst.append(workspace_name)
        continue
    if response.status_code != 200:
        failed_workspaces_lst.append(workspace_name)
        continue

    script_data_results = response.json().get('results')
    for script_data in script_data_results:
        script_name = script_data.get("name", "N/A")
        script_code = script_data.get("code") or ""
        script_author = script_data.get("author", "Unknown")
        script_id = script_data.get("id")

        matched_patterns = find_matches(script_code, TARGET_SCRIPT_CONTENT)

        if matched_patterns:
            number_of_executions = get_number_of_executions(
                script_id, stat_start_datetime, base_url, headers, timeout=120  # Increased timeout to 120 seconds
            )

            for pattern in matched_patterns:
                target_workspace_lst.append(
                    f"{workspace_name} - {script_name} ({script_author}) "
                    f"- [{pattern}] executions since {stat_start_datetime}: {number_of_executions}"
                )

target_workspace_lst.sort()
failed_workspaces_lst.sort()
target_script_with_author_lst.sort()

Checking Workspaces: 100%|██████████| 335/335 [43:14<00:00,  7.74s/it]


In [ ]:
len(target_workspace_lst)

1437

In [ ]:
target_workspace_lst

['Byvertek - a55_report_generator (Jonathan Schein/Alex Kirikeza) - [datetime\\.fromisoformat\\(] executions since 2025-04-01T00:00:00-03:00: 0',
 'Byvertek - a55_report_generator (Jonathan Schein/Alex Kirikeza) - [datetime\\.now\\(] executions since 2025-04-01T00:00:00-03:00: 0',
 'Byvertek - add_image_overlay (Alex Kirikeza) - [datetime\\.now\\(] executions since 2025-04-01T00:00:00-03:00: 458',
 'Byvertek - create_update_pi (Sean MacDonald/Alex Kirikeza) - [datetime\\.now\\(] executions since 2025-04-01T00:00:00-03:00: 0',
 'Byvertek - pdf-consolidation (Luis Medellin) - [datetime\\.now\\(] executions since 2025-04-01T00:00:00-03:00: 0',
 'afl-ca - CF_File_Missing_Fix (Santiago Blason) - [datetime\\.now\\(] executions since 2025-04-01T00:00:00-03:00: 1',
 'afl-ca - create_update_pi (Bruno C) - [datetime\\.now\\(] executions since 2025-04-01T00:00:00-03:00: 158',
 'afl-ca - geo_code_action (Bruno C) - [datetime\\.now\\(] executions since 2025-04-01T00:00:00-03:00: 7',
 'afl-ca - impo

In [ ]:
def save_list_to_txt(data_list, output_path):
  """Saves a list of strings to a text file, with each string on a new line.

  Args:
    data_list: A list of strings to save.
    output_path: The path to the output text file.
  """
  with open(output_path, 'w') as f:
    for item in data_list:
      f.write(str(item) + '\\n')

In [ ]:
path = '/content/drive/MyDrive/SHARED/output2.txt'
save_list_to_txt(target_workspace_lst, path)